# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).

## Configuration


In [1]:
import numpy as np
import xarray as xr
from utils.config import get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve("inputs/WHACS/processed/NewSouthWales_WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

<xarray.Dataset> Size: 3GB
Dimensions:                      (time: 394464, seapoint: 26)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 104B 923284 937305 ... 922215
Data variables: (12/42)
    cge                          (time, seapoint) float64 82MB 7.0 9.2 ... 6.1
    dir                          (time, seapoint) float64 82MB 110.7 ... 169.1
    dp                           (time, seapoint) float64 82MB 99.0 ... 181.0
    fp                           (time, seapoint) float64 82MB 0.112 ... 0.106
    hs                           (time, seapoint) float64 82MB 1.47 ... 1.266
    latitude                     (time, seapoint) float32 41MB -26.44 ... -37.31
    ...                           ...
    spr                          (time, seapoint) float64 82MB 34.9 ... 54.2
    t01                          (time, seapoint) float64 82MB 5.44 ... 5.83
    t02                          (time, seapoint) float64 82MB 4.76 5.0 ... 4.73
    uwnd                         (time, seapoint) float64 82MB -4.1 ... -2.7
    vwnd                         (time, seapoint) float64 82MB 6.5 5.0 ... -7.8
    tp                           (time, seapoint) float64 82MB 8.929 ... 9.434
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S

In [2]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]))

In [3]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"])).load()

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 394464, goal: 26)
Coordinates:
  * time      (time) datetime64[ns] 3MB 1979-01-01 ... 2023-12-31T23:00:00
  * goal      (goal) int64 208B 1 2 3 4 5 6 7 8 9 ... 18 19 20 21 22 23 24 25 26
    seapoint  (goal) int64 208B 923284 937305 937042 ... 930128 929894 922215
Data variables: (12/20)
    dir       (time, goal) float64 82MB 110.7 131.0 129.4 ... 178.1 181.0 169.1
    hs        (time, goal) float64 82MB 1.47 1.69 1.734 ... 1.28 1.322 1.266
    pdir0     (time, goal) float64 82MB 153.3 169.5 156.2 ... 18.3 8.9 23.3
    pdir1     (time, goal) float64 82MB 91.7 88.1 85.8 ... 182.1 182.1 179.7
    pdir2     (time, goal) float64 82MB 43.2 33.3 32.3 33.6 ... 61.8 61.4 61.8
    pdir3     (time, goal) float64 82MB nan 189.9 190.7 192.4 ... 27.0 231.4 nan
    ...        ...
    ptp0      (time, goal) float64 82MB 6.94 7.0 6.75 6.83 ... 2.42 2.66 3.05
    ptp1      (time, goal) float64 82MB 9.08 9.0 9.0 9.03 ... 9.45 9.51 9.29
    ptp2      (time, goal) float64 82MB 10.45 10.88 10.88 10.17 ... 9.9 9.92 9.9
    ptp3      (time, goal) float64 82MB nan 11.96 11.97 ... 10.48 12.37 nan
    spr       (time, goal) float64 82MB 34.9 45.6 43.8 44.0 ... 40.6 43.3 54.2
    tp        (time, goal) float64 82MB 8.929 8.333 8.333 ... 9.709 9.901 9.434
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S